In [ ]:
from pathlib import Path
from cbct_tools import plot_parameter, plot_mas_per_mmas_per_frame
import pandas as pd

df = pd.read_excel('accu_table.xlsx')


df

In [ ]:
import plotly.graph_objects as go
import plotly.offline as pio


def cbct_color_map():
    return {
        (360, "Std"):      "#1f77b4",  # C0
        (360, "Hi-Speed"): "#ff7f0e",  # C1
        (360, "Hi-Fi"):    "#2ca02c",  # C2
        (360, "Hi-Res"):   "#d62728",  # C3
        (360, "no match"): "#9467bd",  # C4

        (180, "Std"):      "#8c564b",  # C5
        (180, "Hi-Speed"): "#e377c2",  # C6
        (180, "Hi-Fi"):    "#7f7f7f",  # C7
        (180, "Hi-Res"):   "#bcbd22",  # C8
    }

def color_for(scan, imaging):
    cmap = cbct_color_map()
    return cmap.get((scan, imaging), "#000000")


def plot_DAP(df, imaging_modes=('Std', 'Hi-Fi', 'Hi-Res', 'Hi-Speed'), scan_modes=(180, 360), export_to_browser=False):

    df = df[(df["Imaging"].isin(imaging_modes)) &
            (df["Scan"].isin(scan_modes))].copy()

    hover_lines = []
    for i, col in enumerate(df.columns):
        hover_lines.append(f"{col}: " + "%{customdata[" + str(i) + "]}")
    hovertemplate = "<br>".join(hover_lines) + "<extra></extra>"

    #df = df.sort_values(["Scan", "Imaging", parameter])
    #df["x_inc"] = df.groupby(["Scan", "Imaging"]).cumcount()

    fig = go.Figure()
    groups = [(scan, im) for scan in scan_modes for im in imaging_modes]

    for scan, im in groups:
        tdf = df[(df["Scan"] == scan) & (df["Imaging"] == im)]
        if tdf.empty:
            continue

        fig.add_trace(go.Scatter(
            x=tdf["mAs"],
            y=tdf['DAP'],
            mode="markers",
            marker=dict(
                size=10,
                color=color_for(scan, im),
                line=dict(color="black", width=0.4)
            ),
            name=f"{scan}, {im}",
            customdata=tdf.values.tolist(),
            hovertemplate=hovertemplate
        ))

    fig.update_layout(
        #title=f"<b>Increasing {parameter}</b> — Modes: {imaging_modes}, Scans: {scan_modes}",
        #xaxis_title=f"Index (increasing {parameter})",
        #yaxis_title=parameter,
        template="plotly_white",
        hovermode="closest",
        legend_title_text="Scan, Imaging"
    )

    if export_to_browser:
        pio.show(fig, renderer="browser")
    else:
        fig.show()

plot_DAP(df=df)

In [ ]:
# 360, Hi-Speed 10 mA, 105,0 mAs, 476 DAP (Lägre risk för pat rörelser)
# 360 std, 6 mA, 105,0 mAs, 419 DAP (lägre dos)

# tre fyra exponeringar i rad,  
# 9 mot 10,5

# 360 std 85, 7,  mAs 122.5, dap 489
# 360 HS  85, 10, mAs 105,   dap 476



# lågdosprotokoll:
# high speed 180 10mA, 54 mAs, mycket bra bild!

# highspeed 180; 54
